# Course 16: Algorithms Behind Modern AI
## Session 10: Nearest neighbours II: HNSW, IVF and product quantization

**Goal:** build a graph index, an inverted-file index and a product quantizer for vector search from scratch, and measure recall against distance computations and bytes per vector for each.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. By hand first: a greedy walk on a six-node graph
Session 9 ended with a problem: in high dimensions an exact index examines almost every point. The indexes of this session are **approximate**. They return points that are near the query, usually the nearest ones, and nothing guarantees it. So every result in this notebook comes with its **recall**: the share of the true nearest neighbours that the index returned.

The first idea is a **proximity graph**: every point is a node, and it is linked to some points near it. To search, start at some node and **walk greedily**: compute the distance from the query to every neighbour of the current node, move to the closest one, and stop when no neighbour is closer than the node you are on.

Draw this graph on squared paper: A (0, 3), B (2, 4), C (2, 1), D (4, 5), E (5, 1), F (6, 3), with the links A–B, A–C, B–C, B–D, C–E and E–F. Always start at A.

* The query is (1, 5). Compute the squared distances you need and follow the walk. Where does it stop? Is that the nearest node?
* The query is (6, 4). Where does the walk stop now? Which node is really the nearest? Which link is missing?

In [ ]:
import heapq, math, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

t_start = time.time()

names = "ABCDEF"
toy_points = np.array([[0, 3], [2, 4], [2, 1], [4, 5], [5, 1], [6, 3]], dtype=float)
toy_links = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1, 4], 3: [1], 4: [2, 5], 5: [4]}       # A-B, A-C, B-C, B-D, C-E, E-F

def greedy_walk(q, start, links, points):
    # Move to the neighbour that is closest to the query; stop when no neighbour is closer than the current node.
    seen = {start: float(((points[start] - q) ** 2).sum())}  # node -> squared distance to the query
    node, path = start, [start]
    while True:
        for nb in links[node]:
            if nb not in seen:
                seen[nb] = float(((points[nb] - q) ** 2).sum())
        best = min(links[node], key=lambda nb: (seen[nb], nb))
        if seen[best] >= seen[node]:
            return path, seen
        node = best
        path.append(node)

for q in [np.array([1.0, 5.0]), np.array([6.0, 4.0])]:
    d2 = ((toy_points - q) ** 2).sum(axis=1)
    path, seen = greedy_walk(q, 0, toy_links, toy_points)
    nearest = int(d2.argmin())
    print(f"query ({q[0]:.0f}, {q[1]:.0f}): squared distances " + ", ".join(f"{n} {d:.0f}" for n, d in zip(names, d2)))
    print("  greedy walk from A: " + " -> ".join(f"{names[n]} ({seen[n]:.0f})" for n in path) + f";  distances computed: {len(seen)} of 6")
    print(f"  the nearest node is {names[nearest]} ({d2[nearest]:.0f}): "
          + ("found" if path[-1] == nearest else f"missed. The walk is stuck at {names[path[-1]]}, a local minimum"))

### 2. The data and the ground truth
The data are **synthetic**: 20,000 vectors with 32 coordinates, stored as float32 like real embeddings (128 bytes per vector). They form 40 overlapping clusters: each coordinate is a cluster centre drawn from a standard normal distribution plus standard normal noise, so half of the variance is between clusters and half is inside them. The 200 queries are new points drawn the same way. Real collections hold millions of vectors with hundreds or thousands of coordinates.

The **ground truth** is the exact top 10 of every query by brute force: 20,000 distance computations per query. We use the squared Euclidean distance everywhere; it gives the same order as the distance itself. Every index below is judged by

* **recall@10:** the share of the true top 10 that it returns;
* **work:** the number of distance computations per query, an exact count that does not depend on the machine;
* **memory:** bytes per vector.

In [ ]:
N, D, N_CLUSTERS, N_QUERIES, K = 20_000, 32, 40, 200, 10
rng = np.random.default_rng(10)
centres = rng.normal(size=(N_CLUSTERS, D))
X = (centres[rng.integers(N_CLUSTERS, size=N)] + rng.normal(size=(N, D))).astype(np.float32)
Q = (centres[rng.integers(N_CLUSTERS, size=N_QUERIES)] + rng.normal(size=(N_QUERIES, D))).astype(np.float32)

t0 = time.perf_counter()
exact = np.array([((X - q) ** 2).sum(axis=1) for q in Q])    # brute force: 200 x 20,000 squared distances
truth = np.argsort(exact, axis=1, kind="stable")[:, :K]
t_brute = (time.perf_counter() - t0) / N_QUERIES
truth_sets = [set(row.tolist()) for row in truth]
ranked = np.sqrt(np.sort(exact, axis=1))

print(f"{N:,d} vectors, {D} coordinates, float32: {X.nbytes // N} bytes per vector, {X.nbytes:,d} bytes in total")
print(f"distance from a query (median over the {N_QUERIES} queries): nearest {np.median(ranked[:, 0]):.2f}, 10th nearest {np.median(ranked[:, 9]):.2f}, "
      f"100th {np.median(ranked[:, 99]):.2f}, median point {np.median(ranked[:, N // 2]):.2f}, farthest {np.median(ranked[:, -1]):.2f}")
print(f"brute force: {N:,d} distance computations per query, recall 1.000")
print(f"  time per query, brute force with NumPy: {1000 * t_brute:.2f} ms")

### 3. A navigable small-world graph, and best-first search
**Building the graph** (Malkov & Yashunin 2020 describe this earlier design, the navigable small world, as their starting point). Insert the points one at a time. For each new point, search the graph built so far for its nearest points, and link the new point to the $M$ nearest that the search found, in both directions. The first points get linked to points that later turn out to be far away. These early long links work like motorways: a search can cross the whole data set in a few steps (Kleinberg 2000 studies when greedy routing on such "small-world" graphs is fast).

**Searching** has to deal with the local minimum of section 1. Instead of one current node we keep the best `ef` nodes found so far, and two heaps (session 7):

* `candidates`, a min-heap: the closest node that we have not expanded yet is on top;
* `results`, a max-heap of at most `ef` nodes: the worst of the best is on top, ready to be replaced.

Take the closest candidate. If it is farther than the worst result, stop. Otherwise compute the distance to each of its neighbours that we have not seen (one NumPy call for all of them), and put every neighbour that beats the worst result into both heaps. With `ef` = 1 this is the greedy walk. A larger `ef` costs more distance computations and escapes more local minima.

The cell first runs this search on the six-node graph, then builds the graph on the 20,000 vectors with $M$ = 8 and `ef` = 40 during the build, and sweeps `ef` from 10 to 200 for the queries. The graph code is written once, with options that section 4 uses: several layers, a limit on the links per node, and a second rule for choosing links.

In [ ]:
def search_layer(q, start, ef, links, vectors, counter):
    # Best-first search in one layer. start: list of (squared distance, node). counter[0] counts distance computations.
    # Returns the ef nearest nodes that were found, as (squared distance, node), nearest first.
    visited = {node for _, node in start}
    candidates = list(start); heapq.heapify(candidates)                      # min-heap: closest unexpanded node on top
    results = [(-d, node) for d, node in start]; heapq.heapify(results)      # max-heap: worst of the best ef on top
    while candidates:
        d, node = heapq.heappop(candidates)
        if d > -results[0][0]:                               # the closest candidate is worse than every result: stop
            break
        new = [nb for nb in links[node] if nb not in visited]
        if not new:
            continue
        visited.update(new)
        dist = ((vectors[new] - q) ** 2).sum(axis=1)         # one NumPy call for all the new neighbours
        counter[0] += len(new)
        for d_nb, nb in zip(dist.tolist(), new):
            if len(results) < ef or d_nb < -results[0][0]:
                heapq.heappush(candidates, (d_nb, nb))
                heapq.heappush(results, (-d_nb, nb))
                if len(results) > ef:
                    heapq.heappop(results)                   # drop the worst result
    return sorted((-d, node) for d, node in results)

q = np.array([6.0, 4.0])
for ef in [1, 2, 3]:
    counter = [1]                                            # the distance to the start node A
    found = search_layer(q, [(float(((toy_points[0] - q) ** 2).sum()), 0)], ef, toy_links, toy_points, counter)
    print(f"six-node graph, query (6, 4), ef = {ef}: returns " + ", ".join(f"{names[n]} ({d:.0f})" for d, n in found)
          + f";  distances computed: {counter[0]} of 6")

class GraphIndex:
    def __init__(self, vectors, M, ef_construction, levels, max_links=None, diverse=False):
        # levels[i]: the top layer of point i (all 0 gives a flat graph).
        # max_links: None for no limit, or (limit in layer 0, limit in the layers above). diverse: the link rule of section 4.
        self.vectors, self.M, self.ef_construction = vectors, M, ef_construction
        self.levels, self.max_links, self.diverse = levels, max_links, diverse
        self.links = [dict() for _ in range(max(levels) + 1)]                # links[layer][node] = list of neighbours
        self.entry, self.top = None, -1                      # the node where every search starts, and its top layer
        self.n_dist = [0]                                    # distance computations during the build

    def select(self, found, m):
        # found: (squared distance to the base node, node), nearest first. Returns at most m of them.
        if not self.diverse:
            return found[:m]                                 # the simple rule: the m nearest
        ids = [node for _, node in found]
        C = self.vectors[ids]
        between = ((C[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).tolist()    # distances between the candidates
        self.n_dist[0] += len(ids) * (len(ids) - 1) // 2
        kept = []
        for a in range(len(ids)):                            # keep a candidate only if it is closer to the base node
            if all(between[a][b] > found[a][0] for b in kept):                   # than to every neighbour kept so far
                kept.append(a)
                if len(kept) == m:
                    break
        return [found[a] for a in kept]

    def shrink(self, node, layer):
        # If a node has more links than allowed, choose again among its neighbours.
        limit = None if self.max_links is None else self.max_links[0 if layer == 0 else 1]
        neighbours = self.links[layer][node]
        if limit is None or len(neighbours) <= limit:
            return
        dist = ((self.vectors[neighbours] - self.vectors[node]) ** 2).sum(axis=1)
        self.n_dist[0] += len(neighbours)
        order = np.argsort(dist, kind="stable").tolist()
        self.links[layer][node] = [nb for _, nb in self.select([(float(dist[o]), neighbours[o]) for o in order], limit)]

    def insert(self, i):
        q, level = self.vectors[i], self.levels[i]
        for layer in range(level + 1):
            self.links[layer][i] = []
        if self.entry is None:                               # the first point
            self.entry, self.top = i, level
            return
        self.n_dist[0] += 1
        start = [(float(((self.vectors[self.entry] - q) ** 2).sum()), self.entry)]
        for layer in range(self.top, level, -1):             # layers above the new point's top layer: greedy walk
            start = search_layer(q, start, 1, self.links[layer], self.vectors, self.n_dist)
        for layer in range(min(level, self.top), -1, -1):    # the new point's layers: search, then link
            found = search_layer(q, start, self.ef_construction, self.links[layer], self.vectors, self.n_dist)
            for d, nb in self.select(found, self.M):
                self.links[layer][i].append(nb)
                self.links[layer][nb].append(i)              # links go both ways
                self.shrink(nb, layer)
            start = found
        if level > self.top:
            self.entry, self.top = i, level

    def search(self, q, k, ef, counter, descend=True):
        counter[0] += 1
        start = [(float(((self.vectors[self.entry] - q) ** 2).sum()), self.entry)]
        if descend:
            for layer in range(self.top, 0, -1):             # upper layers: greedy walk towards the query
                start = search_layer(q, start, 1, self.links[layer], self.vectors, counter)
        return [node for _, node in search_layer(q, start, max(ef, k), self.links[0], self.vectors, counter)[:k]]

def build_graph(name, **options):
    t0 = time.perf_counter()
    g = GraphIndex(X, **options)
    for i in range(N):
        g.insert(i)
    seconds = time.perf_counter() - t0
    degree = [len(neighbours) for neighbours in g.links[0].values()]
    n_links = sum(len(neighbours) for layer in g.links for neighbours in layer.values())
    print(f"{name}: nodes per layer " + ", ".join(f"{len(layer):,d}" for layer in g.links)
          + f"; links per node in layer 0: mean {np.mean(degree):.1f}, largest {max(degree)}")
    print(f"  {n_links:,d} links stored as 4-byte ids: {4 * n_links / N:.1f} bytes per vector on top of the {4 * D};  "
          f"distance computations during the build: {g.n_dist[0]:,d}")
    print(f"  time to build: {seconds:.1f} s")
    return g, 4 * D + 4 * n_links / N

EFS = [10, 15, 20, 30, 50, 100, 200]

def sweep(name, g, descend=True):
    rows = []
    for ef in EFS:
        counter, hits = [0], 0
        t0 = time.perf_counter()
        for qi in range(N_QUERIES):
            hits += len(truth_sets[qi].intersection(g.search(Q[qi], K, ef, counter, descend)))
        rows.append((ef, hits / (K * N_QUERIES), counter[0] / N_QUERIES, 1000 * (time.perf_counter() - t0) / N_QUERIES))
    print(name)
    print("  ef                    | " + " | ".join(f"{ef:7d}" for ef, r, c, ms in rows))
    print("  recall@10             | " + " | ".join(f"{r:7.3f}" for ef, r, c, ms in rows))
    print("  distance computations | " + " | ".join(f"{c:7.1f}" for ef, r, c, ms in rows))
    print("  time per query in ms  | " + " | ".join(f"{ms:7.2f}" for ef, r, c, ms in rows))
    return rows

M = 8
nsw, nsw_bytes = build_graph("flat graph (navigable small world)", M=M, ef_construction=40, levels=[0] * N)
nsw_rows = sweep("flat graph, search from the first point inserted", nsw)

### 4. HNSW: layers, a limit on the links, and a better choice of links
The flat graph of section 3 has two weaknesses. Its motorways exist by accident, and the nodes that carry them collect a very large number of links (look at "largest" above), so expanding one of them costs many distance computations. **HNSW**, hierarchical navigable small world (Malkov & Yashunin 2020), changes three things:

1. **Layers, as in a skip list** (Pugh 1990). Every point gets a random top layer: layer 0 with probability $1 - 1/M$, and each higher layer is $M$ times rarer. Layer 0 holds all points; the upper layers hold fewer and fewer, so their links are long. A search starts at the top, walks greedily in each layer (`ef` = 1), and uses the node it reaches as the start in the layer below. Only layer 0 is searched with the full `ef`.
2. **At most $M$ links per node** in the upper layers and $2M$ in layer 0. When a node gets too many links, it chooses again.
3. **A choice that keeps links in different directions.** Go through the candidates from the nearest to the farthest, and keep a candidate only if it is closer to the node than to every neighbour already kept. A candidate that sits right behind a kept neighbour is dropped: it can be reached through that neighbour. This keeps a few longer links that hold clusters together.

The cell builds HNSW and repeats the sweep. Then two controlled comparisons, each changing one thing:

* **without the descent:** the same graph, but every search starts in layer 0 at the entry node, without the walk through the upper layers;
* **nearest rule:** the same layers and limits, but a node keeps simply its nearest links (the rule of section 3).

To compare at equal recall, we interpolate the number of distance computations between the two sweep points on either side of recall 0.95 and 0.99.

In [ ]:
level_rng = np.random.default_rng(11)
levels = np.floor(-np.log(1.0 - level_rng.random(N)) / math.log(M)).astype(int).tolist()     # P(level >= l) = M ** -l

hnsw, hnsw_bytes = build_graph("HNSW", M=M, ef_construction=40, levels=levels, max_links=(2 * M, M), diverse=True)
hnsw_rows = sweep("HNSW", hnsw)
flat_rows = sweep("HNSW graph, without the descent (start in layer 0 at the entry node)", hnsw, descend=False)
plain, plain_bytes = build_graph("HNSW with the nearest rule", M=M, ef_construction=40, levels=levels, max_links=(2 * M, M), diverse=False)
plain_rows = sweep("HNSW with the nearest rule", plain)

def cost_at(rows, target):
    # distance computations at a given recall, by linear interpolation between two sweep points
    recalls, costs = [r for ef, r, c, ms in rows], [c for ef, r, c, ms in rows]
    if max(recalls) < target:
        return "not reached"
    j = next(j for j, r in enumerate(recalls) if r >= target)
    if j == 0:
        return f"at most {costs[0]:.0f}"
    return f"{costs[j - 1] + (costs[j] - costs[j - 1]) * (target - recalls[j - 1]) / (recalls[j] - recalls[j - 1]):.0f}"

GRAPHS = [("flat graph (section 3)", nsw_rows), ("HNSW", hnsw_rows), ("HNSW graph without the descent", flat_rows),
          ("HNSW with the nearest rule", plain_rows)]
print("\ndistance computations per query at equal recall (interpolated) | recall 0.95 | recall 0.99")
for name, rows in GRAPHS:
    print(f"  {name:60s} | {cost_at(rows, 0.95):>11s} | {cost_at(rows, 0.99):>11s}")

fig, ax = plt.subplots(figsize=(8, 3.4))
for (name, rows), style in zip(GRAPHS, ["s-", "o-", "o--", "^:"]):
    ax.plot([c for ef, r, c, ms in rows], [r for ef, r, c, ms in rows], style, label=name)
ax.axhline(1.0, color="gray", lw=0.5)
ax.set_xscale("log"); ax.set_xlabel(f"distance computations per query (brute force: {N:,d})"); ax.set_ylabel("recall@10")
ax.set_title("graph indexes: recall against work, ef from 10 to 200"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### 5. An inverted file (IVF): search only the nearest cells
A different idea: **partition** the data. Run k-means (here scikit-learn's `KMeans`) to get `nlist` = 128 centroids, and store with each centroid the list of the vectors that are closest to it: its **cell**. The structure is called an inverted file because it maps a cell to its members, like the index at the back of a book.

To search, compute the distance from the query to the 128 centroids, take the `nprobe` nearest cells, and scan only the vectors in them. With `nprobe` = 1 we scan about 1% of the data, but a true neighbour that lies just across a cell border is lost. Raising `nprobe` raises recall and work together; `nprobe` = `nlist` is brute force.

In [ ]:
NLIST = 128
t0 = time.perf_counter()
coarse = KMeans(n_clusters=NLIST, n_init=1, random_state=0).fit(X)
centroids = coarse.cluster_centers_.astype(np.float32)
cells = [np.flatnonzero(coarse.labels_ == c) for c in range(NLIST)]
t_train = time.perf_counter() - t0
cell_sizes = [len(c) for c in cells]
print(f"IVF: {NLIST} cells with {min(cell_sizes)} to {max(cell_sizes)} vectors (mean {np.mean(cell_sizes):.0f})")
print(f"  time to train k-means and fill the cells: {t_train:.1f} s")

ivf_rows = []
print("nprobe | vectors scanned per query | share of the data | distance computations (with the 128 centroids) | recall@10")
for nprobe in [1, 2, 4, 8, 16, 32]:
    hits = scanned = 0
    for qi in range(N_QUERIES):
        q = Q[qi]
        nearest_cells = np.argsort(((centroids - q) ** 2).sum(axis=1), kind="stable")[:nprobe]
        ids = np.concatenate([cells[c] for c in nearest_cells])
        found = ids[np.argsort(((X[ids] - q) ** 2).sum(axis=1), kind="stable")[:K]]
        hits += len(truth_sets[qi].intersection(found.tolist())); scanned += len(ids)
    ivf_rows.append((nprobe, hits / (K * N_QUERIES), NLIST + scanned / N_QUERIES))
    print(f"{nprobe:6d} | {scanned / N_QUERIES:25,.1f} | {scanned / N_QUERIES / N:16.2%}  | {NLIST + scanned / N_QUERIES:45,.1f} | {hits / (K * N_QUERIES):.3f}")

### 6. Product quantization: a vector in 8 bytes
The graph and the inverted file make the search faster, but they still store every vector in full: 128 bytes here, 3,072 bytes for a 768-dimensional embedding. **Product quantization** (PQ; Jégou, Douze & Schmid 2011) compresses the vectors themselves.

* Cut each vector into $m$ parts (here $m$ = 8 parts of 4 coordinates).
* For each part, run k-means with 256 centroids on that part of all vectors: a **codebook**.
* Replace each part by the number of its nearest centroid, one byte. The vector is now $m$ bytes: its **code**.

With 256 centroids per part, the codes can describe $256^m$ different points, far more than any single k-means could store.

**Distances without decompressing.** For a query, compute the squared distance from each of its $m$ parts to the 256 centroids of that part: a table of $m \times 256$ numbers. The approximate distance to any stored vector is then the sum of $m$ table entries, picked by its code. No multiplication is needed per vector.

The approximation mixes up near neighbours, so the usual recipe has two stages: take a **short list** (here the best 100 by approximate distance) and **re-rank** it with the exact vectors, which can stay on disk. The cell measures recall@10 from the codes alone and after re-ranking, for $m$ = 4, 8 and 16.

In [ ]:
def train_pq(X, m):
    width = X.shape[1] // m                                  # coordinates per part
    codebooks = np.zeros((m, 256, width), dtype=np.float32)
    codes = np.zeros((len(X), m), dtype=np.uint8)
    for j in range(m):
        km = KMeans(n_clusters=256, n_init=1, max_iter=25, random_state=j).fit(X[:, j * width:(j + 1) * width])
        codebooks[j], codes[:, j] = km.cluster_centers_, km.labels_
    return codebooks, codes

def pq_distances(q, codebooks, codes):
    m, _, width = codebooks.shape
    table = ((codebooks - q.reshape(m, 1, width)) ** 2).sum(axis=2)          # m x 256: each part of q against each centroid
    return table[np.arange(m), codes].sum(axis=1)            # for every stored vector: add m table entries

SHORT_LIST = 100
tenth = float(np.median(np.sort(exact, axis=1)[:, 9]))       # squared distance to the 10th neighbour
pq_rows = []
print("  m | bytes per vector | compression | mean squared error of a code | recall@10, codes alone | true top 10 in the short list of 100 | recall@10 after re-ranking")
for m in [4, 8, 16]:
    t0 = time.perf_counter()
    codebooks, codes = train_pq(X, m)
    t_train = time.perf_counter() - t0
    width = D // m
    rebuilt = np.concatenate([codebooks[j][codes[:, j]] for j in range(m)], axis=1)
    error = float(((X - rebuilt) ** 2).sum(axis=1).mean())
    hits_codes = hits_short = hits_rerank = 0
    t0 = time.perf_counter()
    for qi in range(N_QUERIES):
        approx = pq_distances(Q[qi], codebooks, codes)
        order = np.argsort(approx, kind="stable")[:SHORT_LIST]
        hits_codes += len(truth_sets[qi].intersection(order[:K].tolist()))
        hits_short += len(truth_sets[qi].intersection(order.tolist()))
        best = order[np.argsort(((X[order] - Q[qi]) ** 2).sum(axis=1), kind="stable")[:K]]       # exact distances for 100 vectors
        hits_rerank += len(truth_sets[qi].intersection(best.tolist()))
    t_query = (time.perf_counter() - t0) / N_QUERIES
    total = K * N_QUERIES
    pq_rows.append((m, hits_codes / total, hits_rerank / total))
    print(f" {m:2d} | {m:16d} | {4 * D // m:10d}x | {error:28.2f} | {hits_codes / total:22.3f} | {hits_short / total:36.3f} | {hits_rerank / total:.3f}")
    print(f"      time: training {t_train:.1f} s, {1000 * t_query:.2f} ms per query")
print(f"for scale: the squared distance from a query to its 10th neighbour is {tenth:.1f} (median)")
print(f"codebooks for m = 8: {8 * 256 * (D // 8) * 4:,d} bytes in total, shared by all vectors")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot([c / N for n, r, c in ivf_rows], [r for n, r, c in ivf_rows], "o-")
for nprobe, r, c in ivf_rows:
    ax[0].annotate(f"{nprobe}", (c / N, r), textcoords="offset points", xytext=(4, -10), fontsize=7)
ax[0].set_xscale("log"); ax[0].set_xlabel("distance computations per query (share of brute force)"); ax[0].set_ylabel("recall@10")
ax[0].set_title("IVF: the labels are nprobe")
pos = np.arange(len(pq_rows))
ax[1].bar(pos - 0.2, [r for m, r, rr in pq_rows], 0.4, label="codes alone")
ax[1].bar(pos + 0.2, [rr for m, r, rr in pq_rows], 0.4, label="after re-ranking 100")
ax[1].set_xticks(pos); ax[1].set_xticklabels([f"{m} bytes" for m, r, rr in pq_rows])
ax[1].set_xlabel("code size per vector"); ax[1].set_ylabel("recall@10"); ax[1].set_title("product quantization"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

### 7. One table
For each index we take the cheapest setting of its sweep that reaches recall 0.95 (if any), and put work and memory side by side. "Distance computations" are exact distances between full vectors. PQ computes an approximate distance for every stored vector (a sum of $m$ table entries), so its rows count exact and approximate distances separately.

In [ ]:
def first_reaching(rows, target=0.95):
    return next((row for row in rows if row[1] >= target), None)

def table_row(method, setting, recall, work, memory):
    print(f"{method:30s} | {setting:23s} | {recall:9.3f} | {work:>31s} | {memory}")

print("method                         | setting                 | recall@10 | distance computations per query | bytes per vector")
table_row("brute force", "-", 1.0, f"{N:,d}", f"{4 * D}")
for name, rows, nbytes in [("flat graph (section 3)", nsw_rows, nsw_bytes), ("HNSW", hnsw_rows, hnsw_bytes),
                           ("HNSW with the nearest rule", plain_rows, plain_bytes)]:
    row = first_reaching(rows)
    if row is None:
        print(f"{name:30s} | no setting of the sweep reaches recall 0.95")
    else:
        table_row(name, f"M = {M}, ef = {row[0]}", row[1], f"{row[2]:,.1f}", f"{nbytes:.1f}")
row = first_reaching(ivf_rows)
if row is None:
    print(f"{'IVF':30s} | no setting of the sweep reaches recall 0.95")
else:
    table_row("IVF", f"nlist = {NLIST}, nprobe = {row[0]}", row[1], f"{row[2]:,.1f}", f"{4 * D + 4} (the vector and a 4-byte id)")
for m, r_codes, r_rerank in pq_rows:
    table_row("PQ, codes alone", f"m = {m}", r_codes, f"0 exact, {N:,d} approximate", f"{m}")
    table_row("PQ, then re-rank 100", f"m = {m}", r_rerank, f"{SHORT_LIST} exact, {N:,d} approximate", f"{m} in memory, {4 * D} on disk")

### 8. The same indexes with real libraries (NOT RUN here)
FAISS (Johnson, Douze & Jégou 2021; Douze et al. 2024) and hnswlib (from the authors of HNSW) are two widely used open-source libraries for vector search. The cell follows their documentation as checked in November 2026; check the current documentation before you run it. Both are written in C++ and are much faster than our Python loops, but the parameters are the ones you have just built: `M`, `efConstruction`, `efSearch`, `nlist`, `nprobe`, and `m` bytes per code.

In [ ]:
# NOT RUN: needs `pip install faiss-cpu hnswlib` (two libraries that are not installed here).
import faiss, hnswlib

flat = faiss.IndexFlatL2(D)                                  # exact search (squared L2 distances)
flat.add(X)
_, truth_faiss = flat.search(Q, K)

graph = faiss.IndexHNSWFlat(D, 16)                           # HNSW with M = 16
graph.hnsw.efConstruction = 100                              # set before add
graph.add(X)
graph.hnsw.efSearch = 50                                     # can be changed for every search
_, ids_graph = graph.search(Q, K)

quantizer = faiss.IndexFlatL2(D)                             # the index that finds the nearest centroids
ivf = faiss.IndexIVFFlat(quantizer, D, 128)                  # nlist = 128 cells
ivf.train(X)                                                 # k-means
ivf.add(X)
ivf.nprobe = 8                                               # the default is 1
_, ids_ivf = ivf.search(Q, K)

quantizer_pq = faiss.IndexFlatL2(D)
ivfpq = faiss.IndexIVFPQ(quantizer_pq, D, 128, 8, 8)         # 128 cells, m = 8 parts, 8 bits per part
ivfpq.train(X)
ivfpq.add(X)
ivfpq.nprobe = 8
_, short_list = ivfpq.search(Q, 100)                         # then re-rank the 100 with the exact vectors, as in section 6

for name, ids in [("HNSW", ids_graph), ("IVF", ids_ivf)]:
    print(name, "recall@10:", np.mean([len(set(a) & set(b)) / K for a, b in zip(ids, truth_faiss)]))

index = hnswlib.Index(space="l2", dim=D)                     # "l2" is the squared distance; "ip" and "cosine" also exist
index.init_index(max_elements=N, ef_construction=100, M=16)
index.add_items(X, np.arange(N))
index.set_ef(50)                                             # ef for the search; it must be at least k
labels, distances = index.knn_query(Q, k=K)

### 9. Exercises

In [ ]:
# TODO: IVF and PQ together (the IndexIVFPQ of the cell above). Keep the cells of section 5, but scan them with the PQ codes
# of section 6 (m = 8) instead of the exact vectors: for the nprobe nearest cells compute approximate distances from the
# lookup table, keep the best 100, and re-rank those with the exact vectors. Report recall@10, the number of table sums and
# the number of exact distance computations for nprobe = 1, 4 and 16. Real systems encode the residual (the vector minus
# its centroid) instead of the vector: try that as well, and compare the recall from the codes alone.
def ivf_pq_search(q, nprobe):
    return None          # replace with your code: return (ids of the 10 best, table sums, exact distance computations)

r = ivf_pq_search(Q[0], 4)
print("not done yet" if r is None else r)

In [ ]:
# TODO: deleting from a graph. Remove 2,000 random points from the HNSW index of section 4 in two ways:
#   (a) "tombstones": keep the nodes and their links, but never return a deleted point;
#   (b) remove the nodes and every link to them, without repairing the neighbours.
# Compute the new ground truth for the 18,000 points that remain, and sweep ef for both. Which one keeps its recall?
# What would a repair step have to do? (The FAISS documentation says that its HNSW index does not support removal.)
def delete_points(index, ids, tombstones=True):
    return None          # replace with your code

r = delete_points(None, [], tombstones=True)
print("not done yet" if r is None else r)
print(f"runtime so far {time.time() - t_start:.0f} s")

### What to notice
(Written from the saved run. All counts are the same on every run, because the seeds are fixed; only the lines that contain the word "time" change.)

1. **A greedy walk can stop at the wrong node.** For the query (1, 5) the walk A → B ends at the nearest node. For (6, 4) it goes A → B → D and stops at D (squared distance 5), because D's only neighbour is farther away. The nearest node is F (1): D and F are close, but the link between them is missing. Best-first search with `ef` = 2 still returns D. With `ef` = 3 it finds F, after computing all 6 distances instead of 4.
2. **The flat graph already works, and it has hubs.** Linking every new point to the 8 nearest points found gave 16.0 links per node on average, but one node has 298. From `ef` = 10 to 200 the recall rose from 0.840 to 0.999, and the work from 380 to 1,070 distance computations per query. Brute force needs 20,000.
3. **HNSW needed fewer distance computations at equal recall.** At recall 0.95 it needed 371 per query against 516 for the flat graph, and at recall 0.99 it needed 568 against 740 (interpolated between sweep points). It also stores fewer links: 44.4 bytes per vector against 64.0. At equal `ef` the comparison looks different: at `ef` = 50 HNSW has the lower recall (0.969 against 0.984) and does much less work (407 against 645). Compare indexes at equal recall, not at equal settings.
4. **Against the plan's expectation, the layers were the smallest part of that gain.** The same HNSW graph, searched without the descent through the upper layers, needed 399 distance computations at recall 0.95 instead of 371. At every `ef` the descent saved between 24 and 34 distance computations per query, with almost the same recall. A likely reason: with 20,000 points in 32 dimensions, any node is only a few links away from any other, so there is little walking to save. The layers are designed for much larger collections, which this lab does not test.
5. **The rule for choosing links mattered more.** With the same layers and limits but the simple rule (keep the nearest), recall stopped at 0.980 for `ef` = 200 and never reached 0.99; at recall 0.95 this graph needed 485 distance computations against 371. The diverse rule stores fewer links (10.2 per node in layer 0 against 11.7) and finds more. Its build cost more in our code: 27.2 million distance computations against 7.5 million, because we compute every distance between the candidates of each choice.
6. **IVF: one cell is not enough, a few are.** With 128 cells, `nprobe` = 1 scanned 0.91% of the vectors and returned 0.521 of the true neighbours. `nprobe` = 4 scanned 3.19% and returned 0.970; `nprobe` = 8 returned 0.991 for 1,391 distance computations. At a recall near 0.97, IVF needed 766 distance computations and HNSW 407. But IVF stores no links, and its build is one run of k-means.
7. **Product quantization: the codes alone are not enough, the codes plus a short list are.** With $m$ = 8 a vector takes 8 bytes instead of 128. Ranking by the codes alone gave recall 0.526. The mean squared error of a code is 5.37, and the squared distances of the 10th and the 100th neighbour differ only by about 13, so close neighbours get mixed up. But the short list of 100 contained 0.994 of the true top 10, and re-ranking returns exactly what the short list contains: recall 0.994. With 4 bytes the short list held only 0.795; with 16 bytes it held everything, and the codes alone gave 0.857.
8. **One table, no winner.** Near recall 0.96 to 0.97: the flat graph needs 541 distance computations and 192.0 bytes per vector, HNSW 407 and 172.4, IVF 766 and 132. PQ with re-ranking needs 100 exact distance computations and 8 bytes in memory, but it still computes an approximate distance for all 20,000 vectors, and the exact vectors must be stored somewhere. Combining IVF with PQ removes most of that scan (first TODO). In the saved run a graph query took about TIME_GRAPH ms at `ef` = 50 in pure Python, and brute force with NumPy about TIME_BRUTE ms.
9. **What this lab does not show.** The data are 20,000 synthetic vectors in 32 dimensions. Real collections are far larger, and the results change with the data: how clustered it is, its dimension, the similarity that is used. No index here gives a guarantee. Every recall above was measured against brute force on 200 queries; do the same on your own data.